# Bias Check — full-inference / no-hidden

Inference only, no training. Learnt recurrent scaling factors come out
systematically **below** their targets, and more so as the problem gets harder.
This asks whether that shrinkage is where the loss actually wants to be.

The trained student is run forward at two settings:

- **learnt** — the run's final state exactly as trained
- **correct** — the same state with only the recurrent scaling factors replaced by
  their targets

Both share the identical learnt low-rank feedforward block, so this isolates the
recurrent scaling factors.

**Reading it:** if `correct` gives a *lower* loss, the shrinkage is a training
bias — the loss prefers the true values and training isn't reaching them. If
`learnt` is lower, the objective's minimum genuinely sits below the targets.

**Caveat.** The targets are not the true model here: `noise_frac` (per-synapse
weight noise) and `missing_unit_fraction` (structurally removed neurons) are
perturbations no scaling factor can undo. The loss-optimal scaling factors are
therefore legitimately displaced from the targets, and a gap measures how far
mis-specification moves the optimum rather than a pathology on its own.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.visualization import (
    QUALITATIVE_COLORS,
    plot_r2_vs_parameter,
    use_project_style,
)
from connectome_snns.visualization.scaling_factors import SF_PATHWAYS

use_project_style()

In [ ]:
config = load_experiment_config("experiment.toml")
RESULTS_DIR = config["output_dir"]

SF_SOURCES = ["learnt", "correct"]
SF_LABELS = {"learnt": "Learnt", "correct": "Correct (target)"}
SF_COLORS = {s: QUALITATIVE_COLORS[i] for i, s in enumerate(SF_SOURCES)}


## Results

In [ ]:
results = pd.read_csv(RESULTS_DIR / "inference_losses.csv")
sources = [s for s in SF_SOURCES if s in set(results["sf_source"])]
runs = list(dict.fromkeys(results["run"]))


def loss_for(run, source):
    row = results[(results["run"] == run) & (results["sf_source"] == source)]
    return float(row["van_rossum_loss"].iloc[0])


print(f"{len(runs)} reference run(s): {runs}")
results


## Loss at Each Setting

Error bars / the spread panel show variation across evaluation chunks, so a gap
much larger than the spread is a real separation rather than chunk noise. The
dashed line is the minimum loss logged during training, for reference.

In [ ]:
traces = np.load(RESULTS_DIR / "loss_traces.npz")
run = runs[0]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
values = [loss_for(run, s) for s in sources]
errors = [
    float(
        results.loc[
            (results["run"] == run) & (results["sf_source"] == s),
            "van_rossum_loss_std",
        ].iloc[0]
    )
    for s in sources
]
ax.bar(range(len(sources)), values, yerr=errors, capsize=4,
       color=[SF_COLORS[s] for s in sources])
ax.axhline(results["training_final_loss"].iloc[0], color="black", linestyle="--",
           linewidth=1, label="Training final loss (logged)")
ax.set_xticks(range(len(sources)))
ax.set_xticklabels([SF_LABELS[s] for s in sources])
ax.set_ylabel("Van Rossum Loss")
ax.set_title("Mean Loss", fontweight="bold")
ax.legend(fontsize=8)

ax = axes[1]
ax.boxplot([traces[f"{run}__{s}"] for s in sources],
           tick_labels=[SF_LABELS[s] for s in sources])
ax.set_ylabel("Van Rossum Loss")
ax.set_title("Spread Across Evaluation Chunks", fontweight="bold")

fig.suptitle(f"Bias Check \u2014 {run}", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()


## Where the Learnt Scaling Factors Sit

Learnt values normalised by the correct ones. Everything below the dashed line is
shrinkage — the effect being diagnosed.

In [ ]:
sf_data = np.load(RESULTS_DIR / "scaling_factors.npz")
cell_types = [str(n) for n in sf_data["output_cell_type_names"]]

ratio = sf_data[f"{run}__learnt"] / sf_data[f"{run}__correct"]
labels = [f"{src} \u2192 {tgt}" for src in cell_types for tgt in cell_types]
values = [ratio[i, j] for i in range(len(cell_types)) for j in range(len(cell_types))]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, values, color=SF_COLORS["learnt"])
ax.axhline(y=1.0, color="black", linestyle="--", linewidth=1, label="Target")
ax.set_ylabel("Scaling Factor / Target")
ax.set_title("Learnt Recurrent Scaling Factors Relative to Target", fontweight="bold")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

pd.DataFrame({"learnt / target": dict(zip(labels, values))}).round(4)


## Firing Rates

Mean visible firing rate under each setting against the teacher. Systematic over-
or under-shoot points at what the shrinkage is compensating for.

In [ ]:
fig, axes = plt.subplots(1, len(cell_types), figsize=(11, 4))
axes = np.atleast_1d(axes)

for ax, cell_type in zip(axes, cell_types):
    for i, source in enumerate(sources):
        subset = results[results["sf_source"] == source]
        ax.bar(i, subset[f"firing_rate/student_{cell_type}_mean"].mean(),
               color=SF_COLORS[source])
    ax.axhline(results[f"firing_rate/teacher_{cell_type}_mean"].mean(),
               color="black", linestyle="--", linewidth=1, label="Teacher")
    ax.set_xticks(range(len(sources)))
    ax.set_xticklabels([SF_LABELS[s] for s in sources])
    ax.set_ylabel("Mean Firing Rate (Hz)")
    ax.set_title(cell_type.capitalize())
    ax.legend(fontsize=8)

fig.suptitle("Student Firing Rates", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()


## Verdict

In [ ]:
learnt, correct = loss_for(run, "learnt"), loss_for(run, "correct")
a, b = traces[f"{run}__learnt"], traces[f"{run}__correct"]

print(f"loss(learnt)  = {learnt:.4f}")
print(f"loss(correct) = {correct:.4f}")
print(f"ratio learnt/correct = {learnt / correct:.3f}")
print(f"chunk-wise distributions disjoint: {min(a) > max(b) or min(b) > max(a)}")
print(f"mean |SF/target - 1| = {np.mean(np.abs(np.array(values) - 1.0)):.4f}")
print()
if correct < learnt:
    print("Correct scaling factors give a LOWER loss \u2192 the shrinkage is a TRAINING")
    print("BIAS: the loss prefers the targets and training is not reaching them.")
else:
    print("Learnt scaling factors give a LOWER loss \u2192 the objective's minimum")
    print("genuinely sits below the targets. Given noise_frac and")
    print("missing_unit_fraction are un-invertible, this is the loss compensating")
    print("for model mis-specification, not a training failure.")
